## Ассоциативные правила

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import random


Генерация данных транзакций...
Сгенерировано 235 транзакций
Уникальных товаров: 15

Примеры первых 5 транзакций:
Транзакция 1: ['Product_H', 'Product_O', 'Product_D']
Транзакция 2: ['Product_E', 'Product_B']
Транзакция 3: ['Product_D', 'Product_B', 'Product_K']
Транзакция 4: ['Product_I', 'Product_D', 'Product_F', 'Product_O']
Транзакция 5: ['Product_M']



### ============================================
### ЧАСТЬ 1: ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ
### ============================================


In [ ]:

def generate_transaction_data(n_transactions=100, n_unique_items=10, avg_items_per_transaction=3):
    """
    Генерирует синтетические данные транзакций для анализа ассоциативных правил
    
    Параметры:
    - n_transactions: количество транзакций
    - n_unique_items: количество уникальных товаров
    - avg_items_per_transaction: среднее количество товаров в транзакции
    
    Возвращает:
    - Список транзакций (каждая транзакция - список товаров)
    - Список уникальных товаров
    """
    
    # Генерируем названия товаров (например, Product_A, Product_B, ...)
    items = [f"Product_{chr(65 + i)}" for i in range(n_unique_items)]
    
    # Создаем список транзакций
    transactions = []
    
    for i in range(n_transactions):
        # Для каждой транзакции случайным образом выбираем количество товаров
        # Используем распределение Пуассона для более реалистичного распределения
        n_items = max(1, int(random.gauss(avg_items_per_transaction, 1.5)))
        
        # Выбираем случайные товары без повторений
        transaction = random.sample(items, min(n_items, n_unique_items))
        transactions.append(transaction)
    
    # Для создания более интересных правил, добавим несколько "часто вместе покупаемых" товаров
    # Например, Product_A и Product_B часто покупают вместе
    for i in range(20):  # Добавляем 20 специальных транзакций
        special_transaction = ['Product_A', 'Product_B']
        # Добавляем случайные дополнительные товары
        additional_items = random.sample([item for item in items if item not in ['Product_A', 'Product_B']], 
                                       random.randint(0, 2))
        special_transaction.extend(additional_items)
        transactions.append(special_transaction)
    
    # Добавим транзакции, где Product_C часто покупают с Product_D
    for i in range(15):
        special_transaction = ['Product_C', 'Product_D']
        additional_items = random.sample([item for item in items if item not in ['Product_C', 'Product_D']], 
                                       random.randint(0, 2))
        special_transaction.extend(additional_items)
        transactions.append(special_transaction)
    
    return transactions, items

# Генерируем данные
print("Генерация данных транзакций...")
transactions, items = generate_transaction_data(
    n_transactions=200,      # 200 транзакций
    n_unique_items=15,       # 15 уникальных товаров
    avg_items_per_transaction=4  # в среднем 4 товара в транзакции
)

print(f"Сгенерировано {len(transactions)} транзакций")
print(f"Уникальных товаров: {len(items)}")
print("\nПримеры первых 5 транзакций:")
for i, trans in enumerate(transactions[:5]):
    print(f"Транзакция {i+1}: {trans}")



### ============================================
### ЧАСТЬ 2: ПРЕДОБРАБОТКА ДАННЫХ
### ============================================

### Преобразуем данные в формат, подходящий для mlxtend
### TransactionEncoder преобразует список транзакций в one-hot encoding формат


In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# Создаем DataFrame с бинарными значениями (True/False)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nРазмерность DataFrame после one-hot encoding: {df.shape}")
print("\nПервые 5 строк DataFrame (фрагмент):")
print(df.iloc[:5, :5])  # Показываем только первые 5 колонок для наглядности



### ============================================
### ЧАСТЬ 3: ПОИСК ЧАСТЫХ НАБОРОВ (APRIORI)
### ============================================

### Применяем алгоритм Apriori для поиска часто встречающихся наборов товаров
### min_support - минимальная поддержка (частота встречаемости набора)


In [ ]:
frequent_itemsets = apriori(
    df, 
    min_support=0.05,        # Минимальная поддержка 5%
    use_colnames=True,        # Использовать названия колонок вместо индексов
    max_len=None              # Максимальная длина набора (None = без ограничений)
)

print(f"\nНайдено {len(frequent_itemsets)} частых наборов товаров")
print("\nТоп-10 самых частых наборов:")
print(frequent_itemsets.nlargest(10, 'support')[['support', 'itemsets']])



### ============================================
### ЧАСТЬ 4: ГЕНЕРАЦИЯ АССОЦИАТИВНЫХ ПРАВИЛ
### ============================================

### Генерируем ассоциативные правила из частых наборов
### Метрики для оценки правил:
### - confidence (достоверность): вероятность наличия B при наличии A
### - lift (подъем): насколько вероятнее наличие B при наличии A по сравнению со случайностью
### - leverage (влияние): разница между наблюдаемой и ожидаемой частотой
### - conviction (убежденность): мера зависимости правила


In [ ]:
rules = association_rules(
    frequent_itemsets, 
    metric="lift",            # Метрика для сортировки правил
    min_threshold=1.2,        # Минимальный порог для метрики (lift > 1.2)
    num_itemsets=len(frequent_itemsets)
)

print(f"\nСгенерировано {len(rules)} ассоциативных правил")

# Добавим информативные названия для колонок (перевод)
rules_display = rules[['antecedents', 'consequents', 
                       'support', 'confidence', 'lift', 'leverage', 'conviction']].copy()

print("\nТоп-10 правил с наибольшим lift:")
print(rules_display.nlargest(10, 'lift').to_string())


### ============================================
### ЧАСТЬ 5: АНАЛИЗ И ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ
### ============================================


In [ ]:

print("\n" + "="*50)
print("АНАЛИЗ ПОЛУЧЕННЫХ РЕЗУЛЬТАТОВ")
print("="*50)

# Фильтруем правила с высокой достоверностью
strong_rules = rules[(rules['confidence'] > 0.5) & (rules['lift'] > 1.5)]
print(f"\nСильные правила (confidence > 0.5, lift > 1.5): {len(strong_rules)}")

if len(strong_rules) > 0:
    print("\nПримеры сильных правил:")
    for idx, row in strong_rules.head().iterrows():
        antecedents = ', '.join(list(row['antecedents']))
        consequents = ', '.join(list(row['consequents']))
        print(f"\nПравило: [{antecedents}] -> [{consequents}]")
        print(f"  Поддержка (support): {row['support']:.3f} - правило встречается в {row['support']*100:.1f}% транзакций")
        print(f"  Достоверность (confidence): {row['confidence']:.3f} - при покупке {antecedents} в {row['confidence']*100:.1f}% случаев покупают {consequents}")
        print(f"  Lift: {row['lift']:.3f} - покупка {antecedents} увеличивает вероятность покупки {consequents} в {row['lift']:.2f} раз")



### ============================================
### ЧАСТЬ 6: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ (опционально)
### ============================================


In [ ]:

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Создаем график для визуализации правил
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # График 1: Зависимость confidence от support
    axes[0, 0].scatter(rules['support'], rules['confidence'], 
                       c=rules['lift'], cmap='viridis', alpha=0.6)
    axes[0, 0].set_xlabel('Support')
    axes[0, 0].set_ylabel('Confidence')
    axes[0, 0].set_title('Confidence vs Support (цвет = lift)')
    plt.colorbar(axes[0, 0].collections[0], ax=axes[0, 0])
    
    # График 2: Гистограмма распределения lift
    axes[0, 1].hist(rules['lift'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Lift')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Распределение значений lift')
    axes[0, 1].axvline(x=1, color='red', linestyle='--', label='lift = 1')
    axes[0, 1].legend()
    
    # График 3: Топ-10 правил по lift
    top_rules = rules.nlargest(10, 'lift')
    y_pos = range(len(top_rules))
    rules_labels = [f"{', '.join(list(a))} → {', '.join(list(c))}" 
                    for a, c in zip(top_rules['antecedents'], top_rules['consequents'])]
    axes[1, 0].barh(y_pos, top_rules['lift'])
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels(rules_labels, fontsize=8)
    axes[1, 0].set_xlabel('Lift')
    axes[1, 0].set_title('Топ-10 правил по метрике lift')
    
    # График 4: Scatter plot матрицы правил
    scatter = axes[1, 1].scatter(rules['support'], rules['confidence'], 
                                 c=rules['lift'], s=rules['leverage']*1000, 
                                 cmap='plasma', alpha=0.6)
    axes[1, 1].set_xlabel('Support')
    axes[1, 1].set_ylabel('Confidence')
    axes[1, 1].set_title('Правила: размер = leverage')
    plt.colorbar(scatter, ax=axes[1, 1])
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("\nДля визуализации установите matplotlib и seaborn: pip install matplotlib seaborn")



### ============================================
### ЧАСТЬ 7: СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
### ============================================


In [ ]:

# Сохраняем результаты в CSV файлы
frequent_itemsets.to_csv('frequent_itemsets.csv', index=False)
rules.to_csv('association_rules.csv', index=False)

print("\nРезультаты сохранены в файлы:")
print("- frequent_itemsets.csv (частые наборы)")
print("- association_rules.csv (ассоциативные правила)")

# Выводим основную статистику
print("\n" + "="*50)
print("ИТОГОВАЯ СТАТИСТИКА")
print("="*50)
print(f"Всего транзакций: {len(transactions)}")
print(f"Уникальных товаров: {len(items)}")
print(f"Частых наборов найдено: {len(frequent_itemsets)}")
print(f"Ассоциативных правил сгенерировано: {len(rules)}")
print(f"Средняя поддержка правил: {rules['support'].mean():.3f}")
print(f"Средняя достоверность: {rules['confidence'].mean():.3f}")
print(f"Средний lift: {rules['lift'].mean():.3f}")